# Study of prediction & reconstruction performance 

## *Experiences description*

### Experiment 3 - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

We then project the latent representations of the test sets into a common PCA space, and we plot some visualization of this PCA space. 

And finally we perform the same analyzes between distributions then in the second experiment : 
- compute multivariate shift metrics in that latent PCA space;
- analyze the moments of the latent principal components;
- analyze extreme latent scores;
- analyze seasonal shifts in latent PCA space.

### Experiment 4 - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider two different method to align the historical and SSP245 climates ;
- We consider a sliced Wasserstein alignment loss 
- And a adversarial classifier.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Interpretation:
- decreasing the alignment term should reduce latent distribution shift between historical and ssp245, and potentially between historical and other warmer climates;
- this must be balanced against reconstruction quality;
- PCA visualizations help determine whether the latent clouds become more mixed and allow us to calculate the same metrics as before onto our "normalized" PCA space.

### Experiment 5 - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict a given variable field over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

This CERA-like setup extends the invariant AE by adding a precipitation predictor:
- AE input: multivariate samples from historical + ssp245 climates.
- AE losses: reconstruction + latent alignment on the first 48 latent dimensions.
- Predictor: MLP on aligned latent dimensions (historical only) to predict a given variable over all 70 patch points.

Global loss used for AE update:
$$
L_{AE} = L_{rec} + \lambda_{align} L_{align} + \lambda_{pred} L_{pred}
$$

And we perform the predictor update at the same time, to be consistent with the end to end training used in the original CERA architecture.

## Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from IPython.display import display
import seaborn as sns
import json
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error
import pickle
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter1d
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

**Data Loading**

In [ ]:
num_samples = 1000000
type_AE = "CNN" # choose between "MLP" and "CNN"
type_alignment = "swd" # choose between "swd" and "adversarial"
mask_strategy = "central_6" # mask to predict
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
variable = mask_strategy # variable to predict

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v1000")
if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

We import the evaluation DataFrames

In [ ]:
evaluation_root = Path("/glade/work/tsalin/CMIP/model_evaluation")
if not evaluation_root.exists():
    raise FileNotFoundError(f"Model evaluation directory not found: {evaluation_root}")


def _load_quality_payload(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, dict):
        raise TypeError(f"Expected a dict payload in {file_path.name}, got {type(content)}")
    return content


def _load_history_df(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, pd.DataFrame):
        raise TypeError(f"Expected a DataFrame in {file_path.name}, got {type(content)}")
    return content.copy()


# ── exp5 mask CERA ────────────────────────────────────────────────────────────
_cera_prefix  = f"cera_ns{num_samples}_{type_AE}_{type_alignment}_{mask_strategy}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_"
CERA_quality  = evaluation_root / "CERA" / f"{_cera_prefix}quality_df.pkl"
CERA_history  = evaluation_root / "CERA" / f"{_cera_prefix}history_df.pkl"

# ── exp5 mask baseline simple ─────────────────────────────────────────────────
_bs_prefix           = f"baseline_simple_ns{num_samples}_{mask_strategy}_{val_fraction}_{test_fraction}_"
Baseline_simple_quality = evaluation_root / "Baseline_simple" / f"{_bs_prefix}quality_df.pkl"
Baseline_simple_history = evaluation_root / "Baseline_simple" / f"{_bs_prefix}history_df.pkl"

# ── exp5 mask baseline CERA noalign ──────────────────────────────────────────
_noalign_prefix              = f"baseline_CERA_noalign_ns{num_samples}_{type_AE}_{mask_strategy}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_"
Baseline_CERA_noalign_quality = evaluation_root / "Baseline_CERA_noalign" / f"{_noalign_prefix}quality_df.pkl"
Baseline_CERA_noalign_history = evaluation_root / "Baseline_CERA_noalign" / f"{_noalign_prefix}history_df.pkl"


# Load quality payloads
cera_quality_payload                  = _load_quality_payload(CERA_quality)
baseline_simple_quality_payload       = _load_quality_payload(Baseline_simple_quality)
baseline_cera_noalign_quality_payload = _load_quality_payload(Baseline_CERA_noalign_quality)

# Load history DataFrames
cera_history_df                  = _load_history_df(CERA_history)
baseline_simple_history_df       = _load_history_df(Baseline_simple_history)
baseline_cera_noalign_history_df = _load_history_df(Baseline_CERA_noalign_history)


## Part 1 - Evaluation metrics computation

In [ ]:
def _safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def _safe_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size == 0:
        return np.nan
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def _parse_variable_slices(value_names):
    slices = []
    seen = {}
    for i, name in enumerate(value_names):
        var = str(name).split("@")[0]
        if var not in seen:
            seen[var] = i
            slices.append((var, i))
    result = []
    for k, (var, start) in enumerate(slices):
        end = slices[k + 1][1] if k + 1 < len(slices) else len(value_names)
        result.append((var, slice(start, end)))
    return result


def _compute_scores_for_component(meta_df, truth_arr, pred_arr, value_names):
    var_slices = _parse_variable_slices(value_names)
    N = len(meta_df)
    meta_reset = meta_df.reset_index(drop=True)

    # Per-sample R²: uses per-climate column mean as baseline.
    # Per-climate mean avoids division by zero when only one point is masked,
    # and is more rigorous (baseline calibrated on the same climate regime).
    sample_r2_by_var = {}
    for var, sl in var_slices:
        t = truth_arr[:, sl].astype(np.float64)
        p = pred_arr[:, sl].astype(np.float64)
        ss_res = np.nansum((t - p) ** 2, axis=1)
        t_mean_by_sample = np.empty_like(t)
        for scenario, grp in meta_reset.groupby("scenario", sort=False):
            idx = grp.index.to_numpy()
            t_mean_by_sample[idx] = np.nanmean(t[idx], axis=0, keepdims=True)
        ss_tot = np.nansum((t - t_mean_by_sample) ** 2, axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            r2 = np.where(ss_tot > 0, 1.0 - ss_res / ss_tot, np.nan)
        sample_r2_by_var[var] = r2

    var_names = [v for v, _ in var_slices]
    sample_r2_list = [
        {var: float(sample_r2_by_var[var][i]) for var in var_names}
        for i in range(N)
    ]

    # Global R²/RMSE per scenario via numpy array indexing
    global_scores = {}
    for scenario, grp in meta_reset.groupby("scenario", sort=False):
        idx = grp.index.to_numpy()
        r2g, rmseg = {}, {}
        for var, sl in var_slices:
            t_all = truth_arr[idx, sl].ravel().astype(np.float64)
            p_all = pred_arr[idx, sl].ravel().astype(np.float64)
            r2g[var]   = _safe_r2(t_all, p_all)
            rmseg[var] = _safe_rmse(t_all, p_all)
        global_scores[scenario] = {"r2_global": r2g, "rmse_global": rmseg}

    out = meta_reset.copy()
    out["r2"]           = sample_r2_list
    out["r2_global"]    = out["scenario"].map(lambda s: global_scores[s]["r2_global"])
    out["rmse_global"]  = out["scenario"].map(lambda s: global_scores[s]["rmse_global"])
    # Store numpy rows as views — no data duplication
    out["truth_values"] = list(truth_arr)
    out["pred_values"]  = list(pred_arr)
    out["value_names"]  = [list(value_names)] * N
    return out
def add_sample_and_global_scores(payload):
    frames = []

    if "meta_reconstruction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_reconstruction"],
            payload["truth_reconstruction"],
            payload["pred_reconstruction"],
            payload["reconstruction_value_names"],
        ))

    if "meta_prediction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_prediction"],
            payload["truth_prediction"],
            payload["pred_prediction"],
            payload["prediction_value_names"],
        ))

    if not frames:
        raise ValueError("Payload has neither 'meta_reconstruction' nor 'meta_prediction'.")

    return pd.concat(frames, ignore_index=True)

In [ ]:
# Save mask geometry constants from CERA payload (same mask for all exp5 setups)
mask_constants = {
    k: cera_quality_payload[k]
    for k in ("masked_point_indices", "masked_feature_columns",
               "visible_point_indices", "visible_feature_columns")
    if k in cera_quality_payload
}

complete_cera_quality_df = add_sample_and_global_scores(cera_quality_payload)
del cera_quality_payload

complete_baseline_simple_quality_df = add_sample_and_global_scores(baseline_simple_quality_payload)
del baseline_simple_quality_payload

complete_baseline_cera_noalign_quality_df = add_sample_and_global_scores(baseline_cera_noalign_quality_payload)
del baseline_cera_noalign_quality_payload

## Part 2 - Plotting quality results

In [ ]:
reconstruction_sources = [
    ("CERA",              complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
]

setup_order = [
    "CERA", "Baseline no align",
]

climate_offsets = {
    "historical": (0, 0), "ssp245": (0, 1), "ssp370": (1, 0), "ssp585": (1, 1),
}

r2_heatmap_rows = []
variable_order  = []
seen_variables  = set()

for setup_name, setup_df in reconstruction_sources:
    recon_df = setup_df[setup_df["component"] == "reconstruction"].copy()
    if recon_df.empty:
        continue

    for scenario_name, scenario_df in recon_df.groupby("scenario", sort=False):
        climate = str(scenario_name)
        if climate not in climate_offsets:
            continue

        r2_global = scenario_df.iloc[0]["r2_global"]
        if not isinstance(r2_global, dict):
            continue

        for var, r2_val in r2_global.items():
            var = str(var)
            if var not in seen_variables:
                seen_variables.add(var)
                variable_order.append(var)
            r2_heatmap_rows.append({
                "setup": setup_name, "variable": var,
                "climate": climate, "r2_global": r2_val,
            })

if not variable_order:
    raise ValueError("No reconstructed variables found for the heatmap.")

heatmap_matrix = np.full(
    (2 * len(variable_order), 2 * len(setup_order)), np.nan, dtype=float
)
setup_index    = {s: i for i, s in enumerate(setup_order)}
variable_index = {v: i for i, v in enumerate(variable_order)}

for row in r2_heatmap_rows:
    if row["setup"] not in setup_index:
        continue
    si = setup_index[row["setup"]]
    vi = variable_index[row["variable"]]
    cr, cc = climate_offsets[row["climate"]]
    heatmap_matrix[2*vi + cr, 2*si + cc] = row["r2_global"]

finite_vals = heatmap_matrix[np.isfinite(heatmap_matrix)]
if finite_vals.size == 0:
    raise ValueError("No finite r2_global values found for reconstruction heatmap.")

cmap_r2 = plt.get_cmap("Blues").copy()
cmap_r2.set_bad("#FFFFFF")

fig, ax = plt.subplots(
    figsize=(max(11, 1.45 * len(setup_order) + 3),
             max(7, 0.55 * len(variable_order) + 2.5)),
    constrained_layout=True,
)
im = ax.imshow(heatmap_matrix, aspect="auto", cmap=cmap_r2,
               vmin=float(finite_vals.min()), vmax=1.0, origin="upper")

ax.set_xticks(np.arange(len(setup_order)) * 2 + 0.5)
ax.set_xticklabels(setup_order, rotation=25, ha="right")
ax.set_yticks(np.arange(len(variable_order)) * 2 + 0.5)
ax.set_yticklabels(variable_order)
ax.set_xlabel("Setup")
ax.set_ylabel("Reconstructed variable")
ax.set_title(
    "Global $R^2$ for reconstruction\n"
    "Quadrants: historical (top-left), ssp245 (top-right), ssp370 (bottom-left), ssp585 (bottom-right)"
)
ax.set_xticks(np.arange(-0.5, heatmap_matrix.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, heatmap_matrix.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)
for sb in np.arange(2, heatmap_matrix.shape[1], 2):
    ax.axvline(sb - 0.5, color="#111827", linewidth=1.3)
for vb in np.arange(2, heatmap_matrix.shape[0], 2):
    ax.axhline(vb - 0.5, color="#111827", linewidth=1.3)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Global $R^2$")
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP       = "CERA"    # "CERA", "Baseline no align", "exp3 AEh", ...
VIZ_CLIMATE     = "historical"   
VIZ_VARIABLE    = "va500"     # variable to visualize
VIZ_SAMPLE_IDX  = 0
# ─────────────────────────────────────────────────────────────────────────────────────

import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

_catalog_path = None
for _name in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
    _p = precomputed_dir / _name
    if _p.exists():
        _catalog_path = _p
        break
if _catalog_path is None:
    raise FileNotFoundError(f"patch_catalog introuvable dans {precomputed_dir}.")
if _catalog_path.suffix == ".pkl":
    with open(_catalog_path, "rb") as _f:
        _patch_catalog = pickle.load(_f)
else:
    _patch_catalog = pd.read_csv(_catalog_path)


def _get_patch_info(patch_id):
    rows = _patch_catalog.loc[_patch_catalog["patch_id"] == int(patch_id)]
    if rows.empty:
        raise ValueError(f"patch_id={patch_id} introuvable dans patch_catalog.")
    return rows.iloc[0]


_setup_dict_r = dict(reconstruction_sources)
if VIZ_SETUP not in _setup_dict_r:
    raise ValueError(f"Setup '{VIZ_SETUP}' inconnu. Disponibles : {list(_setup_dict_r.keys())}")

_recon_df_r = _setup_dict_r[VIZ_SETUP][
    (_setup_dict_r[VIZ_SETUP]["component"] == "reconstruction") &
    (_setup_dict_r[VIZ_SETUP]["scenario"] == VIZ_CLIMATE)
].reset_index(drop=True)

if _recon_df_r.empty:
    raise ValueError(f"Aucune donnée reconstruction pour setup='{VIZ_SETUP}', climat='{VIZ_CLIMATE}'.")
if VIZ_SAMPLE_IDX >= len(_recon_df_r):
    raise IndexError(f"VIZ_SAMPLE_IDX={VIZ_SAMPLE_IDX} hors limites (max {len(_recon_df_r)-1}).")

_row_r = _recon_df_r.iloc[VIZ_SAMPLE_IDX]
_vnames_r = list(_row_r["value_names"])
_all_vars_r = sorted({str(n).split("@")[0] for n in _vnames_r})
if VIZ_VARIABLE not in _all_vars_r:
    raise ValueError(f"Variable '{VIZ_VARIABLE}' absente. Disponibles : {_all_vars_r}")

# Filter truth and prediction values for the selected variable
_var_col_mask_r = np.array([str(n).split("@")[0] == VIZ_VARIABLE for n in _vnames_r])
_truth_flat_r   = np.asarray(_row_r["truth_values"], dtype=float)[_var_col_mask_r]
_recon_flat_r   = np.asarray(_row_r["pred_values"],  dtype=float)[_var_col_mask_r]

# Spatial positioning of the patch
_patch_id_r = int(_row_r["patch_id"])
_info_r     = _get_patch_info(_patch_id_r)
_n_lat_r    = int(_info_r["lat_stop_idx"] - _info_r["lat_start_idx"])
_n_lon_r    = int(_info_r["lon_stop_idx"] - _info_r["lon_start_idx"])
_lats_r     = np.linspace(float(_info_r["lat_start"]), float(_info_r["lat_stop"]), _n_lat_r)
_lons_r     = np.linspace(float(_info_r["lon_start"]), float(_info_r["lon_stop"]), _n_lon_r)
_lon_grid_r, _lat_grid_r = np.meshgrid(_lons_r, _lats_r)

# Points visible and masked indices (1D) in the patch grid
_vis_idx  = np.asarray(mask_constants.get("visible_point_indices", []), dtype=int)
_mask_idx = np.asarray(mask_constants.get("masked_point_indices",  []), dtype=int)


_n_vis_pts = len(_vis_idx)
_var_local_idx = np.where([str(n).split("@")[0] == VIZ_VARIABLE for n in _vnames_r])[0]

# Coordinate lat/lon of visible points
_vis_rows  = _vis_idx // _n_lon_r
_vis_cols  = _vis_idx %  _n_lon_r
_vis_lats  = _lats_r[_vis_rows]
_vis_lons  = _lons_r[_vis_cols]

_mask_rows = _mask_idx // _n_lon_r
_mask_cols = _mask_idx %  _n_lon_r
_mask_lats = _lats_r[_mask_rows]
_mask_lons = _lons_r[_mask_cols]

_vmin_r = float(min(np.nanmin(_truth_flat_r), np.nanmin(_recon_flat_r)))
_vmax_r = float(max(np.nanmax(_truth_flat_r), np.nanmax(_recon_flat_r)))
_cmap_r = "YlOrRd"
_proj_r = ccrs.PlateCarree()

_lon_min_r = float(_lon_grid_r.min())
_lon_max_r = float(_lon_grid_r.max())
_lat_min_r = float(_lat_grid_r.min())
_lat_max_r = float(_lat_grid_r.max())
_mg_lon = max(1.0, 0.2 * (_lon_max_r - _lon_min_r))
_mg_lat = max(1.0, 0.2 * (_lat_max_r - _lat_min_r))
_extent_r = [_lon_min_r - _mg_lon, _lon_max_r + _mg_lon,
             _lat_min_r - _mg_lat, _lat_max_r + _mg_lat]

fig = plt.figure(figsize=(17, 5.5), constrained_layout=True)
gs  = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[2, 2, 1.4])


def _map_panel_r(pos):
    ax = fig.add_subplot(pos, projection=_proj_r)
    ax.set_extent(_extent_r, crs=_proj_r)
    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.35)
    gl.top_labels = gl.right_labels = False
    return ax


ax_recon_r = _map_panel_r(gs[0])
ax_truth_r = _map_panel_r(gs[1])
ax_world_r = fig.add_subplot(gs[2], projection=_proj_r)

_norm_r = plt.Normalize(vmin=_vmin_r, vmax=_vmax_r)
_cmap_r_obj = plt.get_cmap(_cmap_r).copy()
_cmap_r_obj.set_bad(alpha=0)

# 2D arrays for reconstruction and ground truth, with NaN for masked points
import numpy.ma as _nma
_truth_2d_r = np.full((_n_lat_r, _n_lon_r), np.nan)
_recon_2d_r = np.full((_n_lat_r, _n_lon_r), np.nan)
_truth_2d_r[_vis_rows, _vis_cols] = _truth_flat_r
_recon_2d_r[_vis_rows, _vis_cols] = _recon_flat_r

# reconstruction grid panel
_im_r = ax_recon_r.pcolormesh(
    _lon_grid_r, _lat_grid_r, _nma.masked_invalid(_recon_2d_r),
    shading="auto", cmap=_cmap_r_obj, norm=_norm_r, transform=_proj_r, zorder=4,
)
ax_recon_r.scatter(_mask_lons, _mask_lats, s=40, color="gray",
                   marker="x", linewidths=1.2, transform=_proj_r, zorder=5)
ax_recon_r.set_title(
    f"Reconstruction  ({VIZ_SETUP})\n{VIZ_VARIABLE}  ·  {VIZ_CLIMATE}  ·  sample #{VIZ_SAMPLE_IDX}",
    fontweight="bold",
)

# ground truth panel
ax_truth_r.pcolormesh(
    _lon_grid_r, _lat_grid_r, _nma.masked_invalid(_truth_2d_r),
    shading="auto", cmap=_cmap_r_obj, norm=_norm_r, transform=_proj_r, zorder=4,
)
ax_truth_r.scatter(_mask_lons, _mask_lats, s=40, color="gray",
                   marker="x", linewidths=1.2, transform=_proj_r, zorder=5)
ax_truth_r.set_title(
    f"Ground Truth\n{VIZ_VARIABLE}  ·  {VIZ_CLIMATE}  ·  sample #{VIZ_SAMPLE_IDX}",
    fontweight="bold",
)

fig.colorbar(_im_r, ax=[ax_recon_r, ax_truth_r], orientation="vertical",
             fraction=0.035, pad=0.03, label=VIZ_VARIABLE)

# world map
ax_world_r.set_global()
ax_world_r.coastlines(resolution="110m", linewidth=0.7, color="black")
ax_world_r.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax_world_r.gridlines(linewidth=0.3, alpha=0.35)
ax_world_r.add_patch(mpatches.Rectangle(
    (_lon_min_r, _lat_min_r), _lon_max_r - _lon_min_r, _lat_max_r - _lat_min_r,
    linewidth=2.0, edgecolor="red", facecolor="none", transform=_proj_r, zorder=5,
))
ax_world_r.set_title(f"Localisation du patch\npatch_id = {_patch_id_r}", fontweight="bold")

fig.suptitle(
    f"Reconstruction vs Ground Truth  ·  {VIZ_SETUP}  ·  {VIZ_CLIMATE}  ·  {VIZ_VARIABLE}",
    fontweight="bold", fontsize=13,
)
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_LAT  = "historical"  
VIZ_VARIABLE_LAT = "va500"     # variable to visualize
N_LAT_BINS       = 30
# ─────────────────────────────────────────────────────────────────────────────────────


if "_patch_catalog" not in dir():
    for _n in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
        _pp = precomputed_dir / _n
        if _pp.exists():
            with open(_pp, "rb") as _f:
                _patch_catalog = pickle.load(_f)
            break

_cat_idx_lat = _patch_catalog.set_index("patch_id")

# verify that the variable exists in the reconstruction data for the selected climate
_sample_vars_lat = None
for _sn_l, _sdf_l in reconstruction_sources:
    _tmp_l = _sdf_l[
        (_sdf_l["component"] == "reconstruction") &
        (_sdf_l["scenario"] == VIZ_CLIMATE_LAT)
    ]
    if not _tmp_l.empty and "r2" in _tmp_l.columns:
        _first_r2 = _tmp_l.iloc[0]["r2"]
        if isinstance(_first_r2, dict):
            _sample_vars_lat = sorted(_first_r2.keys())
            break

if _sample_vars_lat is None:
    raise ValueError(f"Aucune donnée reconstruction avec colonne 'r2' pour climat='{VIZ_CLIMATE_LAT}'.")
if VIZ_VARIABLE_LAT not in _sample_vars_lat:
    raise ValueError(f"Variable '{VIZ_VARIABLE_LAT}' absente. Disponibles : {_sample_vars_lat}")

_lat_r2_by_setup = {}
for _sname, _sdf in reconstruction_sources:
    _filt = _sdf[
        (_sdf["component"] == "reconstruction") &
        (_sdf["scenario"] == VIZ_CLIMATE_LAT)
    ]
    if _filt.empty or "patch_id" not in _filt.columns or "r2" not in _filt.columns:
        continue

    _pairs = []
    for _, _row in _filt.iterrows():
        _pid = int(_row["patch_id"])
        if _pid not in _cat_idx_lat.index:
            continue
        _info_l  = _cat_idx_lat.loc[_pid]
        _lat_c   = (float(_info_l["lat_start"]) + float(_info_l["lat_stop"])) / 2.0
        _r2_dict = _row["r2"]
        if isinstance(_r2_dict, dict):
            _r2_val = _r2_dict.get(VIZ_VARIABLE_LAT, np.nan)
            if np.isfinite(float(_r2_val)):
                _pairs.append((_lat_c, float(_r2_val)))
    if _pairs:
        _lat_r2_by_setup[_sname] = _pairs

if not _lat_r2_by_setup:
    raise ValueError("Aucune donnée R² par latitude disponible.")

_all_lats_f = [lat for pairs in _lat_r2_by_setup.values() for lat, _ in pairs]
_lat_bins_p = np.linspace(min(_all_lats_f), max(_all_lats_f), N_LAT_BINS + 1)
_lat_ctrs_p = (_lat_bins_p[:-1] + _lat_bins_p[1:]) / 2

_setup_colors_lat = {
    "CERA": "#2F3B52", "Baseline no align": "#2C7FB8",
    "exp3 AEh": "#E63946", "exp3 AEhs2": "#F4A261",
    "exp3 AEhs2s3": "#2A9D8F", "exp3 AEall": "#6A3D9A", "exp4": "#264653",
}
_dflt_cols = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
for _ci, (_sname, _pairs) in enumerate(_lat_r2_by_setup.items()):
    _la = np.array([p[0] for p in _pairs])
    _ra = np.array([p[1] for p in _pairs])
    _col = _setup_colors_lat.get(_sname, _dflt_cols[_ci % len(_dflt_cols)])
    _mr = np.full(N_LAT_BINS, np.nan)
    _sr = np.full(N_LAT_BINS, np.nan)
    for _bi in range(N_LAT_BINS):
        _m = (_la >= _lat_bins_p[_bi]) & (_la < _lat_bins_p[_bi + 1])
        _n = int(np.sum(_m))
        if _n >= 2:
            _mr[_bi] = np.mean(_ra[_m])
            _sr[_bi] = np.std(_ra[_m], ddof=1)
        elif _n == 1:
            _mr[_bi] = _ra[_m][0]
    _v = np.isfinite(_mr)
    _sc_ = np.where(np.isfinite(_sr[_v]), _sr[_v], 0.0)
    ax.plot(_lat_ctrs_p[_v], _mr[_v], linewidth=1.8, color=_col, label=_sname)
    ax.fill_between(_lat_ctrs_p[_v], _mr[_v] - _sc_, _mr[_v] + _sc_,
                    alpha=0.15, color=_col)

ax.axhline(0.0, color="#6B7280", linewidth=0.9, linestyle="--", alpha=0.7)
ax.set_xlabel("Latitude (°)", fontweight="bold")
ax.set_ylabel(f"$R^2$ ({VIZ_VARIABLE_LAT})", fontweight="bold")
ax.set_title(
    f"$R^2$ reconstruction par latitude  ·  {VIZ_VARIABLE_LAT}  ·  {VIZ_CLIMATE_LAT}\n"
    "(moyenne ± écart-type inter-samples, latitude = centre du patch)",
    fontweight="bold",
)
ax.legend(title="Setup", fontsize=9, loc="best", framealpha=0.9)
ax.grid(alpha=0.25, linestyle=":")
plt.show()

## Part 3 - Plotting quality results - prediction

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_PRED_VARIABLE = "va500"   # variable to visualize for prediction evaluation
# ─────────────────────────────────────────────────────────────────────────────────────


prediction_sources = [
    ("CERA",              complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("Baseline simple",   complete_baseline_simple_quality_df),
]

_pred_setup_order = ["CERA", "Baseline no align", "Baseline simple"]
_pred_markers = {"CERA": "o", "Baseline no align": "s", "Baseline simple": "^"}
_pred_colors  = {"CERA": "#2F3B52", "Baseline no align": "#2C7FB8", "Baseline simple": "#CC7B39"}

# ── R² global per setup ────────────────────────────────────────────────────────────
r2_pred_rows = []
for setup_name, df in prediction_sources:
    pred_df = df[df["component"] == "prediction"]
    if pred_df.empty:
        continue
    for _, row in pred_df.iterrows():
        r2_val = row["r2_global"]
        if isinstance(r2_val, dict):
            r2_val = r2_val.get(VIZ_PRED_VARIABLE, np.nan)
        r2_pred_rows.append({"setup": setup_name, "scenario": row["scenario"], "r2_global": r2_val})

r2_pred_df = pd.DataFrame(r2_pred_rows)
scenario_order_pred = list(dict.fromkeys(r2_pred_df["scenario"].tolist()))

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in _pred_setup_order:
    sub = r2_pred_df[r2_pred_df["setup"] == setup_name].copy()
    if sub.empty:
        continue
    sub["scenario"] = pd.Categorical(sub["scenario"], categories=scenario_order_pred, ordered=True)
    sub = sub.sort_values("scenario")
    ax.plot(sub["scenario"], sub["r2_global"], linestyle="None",
            marker=_pred_markers[setup_name], markersize=7,
            color=_pred_colors[setup_name], label=setup_name)

ax.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.8)
ax.set_xlabel("Scenario")
ax.set_ylabel(f"Global $R^2$ for {VIZ_PRED_VARIABLE}")
ax.set_title(f"Global $R^2$ by scenario and setup\n{VIZ_PRED_VARIABLE} (prediction)")
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=[
    Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
    for s in _pred_setup_order
], title="Setup")
plt.xticks(rotation=20, ha="right")
plt.show()

# ── Delta R² vs historical ────────────────────────────────────────────────────────────
_r2u = r2_pred_df.groupby(["setup", "scenario"], sort=False)["r2_global"].first().reset_index()
_r2p = _r2u.pivot(index="setup", columns="scenario", values="r2_global")
_ref = next((c for c in ("historical", "hist") if c in _r2p.columns), None)
if _ref is not None:
    _ssps = [c for c in _r2p.columns if c != _ref]
    _delta = _r2p[_ssps].subtract(_r2p[_ref], axis=0)
    fig2, ax2 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
    for setup_name in _pred_setup_order:
        if setup_name not in _delta.index:
            continue
        _rd = _delta.loc[setup_name]
        ax2.plot(_rd.index, _rd.values, linestyle="None",
                 marker=_pred_markers[setup_name], markersize=7,
                 color=_pred_colors[setup_name], label=setup_name)
    ax2.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
    ax2.set_xlabel("Scenario")
    ax2.set_ylabel(f"Δ Global $R^2$ ({VIZ_PRED_VARIABLE})")
    ax2.set_title(f"Δ Global $R^2$ vs. {_ref}\n{VIZ_PRED_VARIABLE} (prediction)")
    ax2.grid(axis="y", alpha=0.25)
    ax2.legend(handles=[
        Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
        for s in _pred_setup_order if s in _delta.index
    ], title="Setup")
    plt.xticks(rotation=20, ha="right")
    plt.show()

# ── RMSE global per setup ──────────────────────────────────────────────────────────
rmse_pred_rows = []
for setup_name, df in prediction_sources:
    pred_df = df[df["component"] == "prediction"]
    if pred_df.empty:
        continue
    for _, row in pred_df.iterrows():
        rmse_val = row["rmse_global"]
        if isinstance(rmse_val, dict):
            rmse_val = rmse_val.get(VIZ_PRED_VARIABLE, np.nan)
        rmse_pred_rows.append({"setup": setup_name, "scenario": row["scenario"], "rmse_global": rmse_val})

rmse_pred_df = pd.DataFrame(rmse_pred_rows)

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in _pred_setup_order:
    sub = rmse_pred_df[rmse_pred_df["setup"] == setup_name].copy()
    if sub.empty:
        continue
    sub["scenario"] = pd.Categorical(sub["scenario"], categories=scenario_order_pred, ordered=True)
    sub = sub.sort_values("scenario")
    ax.plot(sub["scenario"], sub["rmse_global"], linestyle="None",
            marker=_pred_markers[setup_name], markersize=7,
            color=_pred_colors[setup_name], label=setup_name)

ax.set_xlabel("Scenario")
ax.set_ylabel(f"Global RMSE ({VIZ_PRED_VARIABLE})")
ax.set_title(f"Global RMSE by scenario and setup\n{VIZ_PRED_VARIABLE} (prediction)")
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=[
    Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
    for s in _pred_setup_order
], title="Setup")
plt.xticks(rotation=20, ha="right")
plt.show()

# ── Ratio RMSE = RMSE(ssp) / RMSE(historical) ────────────────────────────────────────
_rmse_uniq = rmse_pred_df.groupby(["setup", "scenario"], sort=False)["rmse_global"].first().reset_index()
_rmse_piv  = _rmse_uniq.pivot(index="setup", columns="scenario", values="rmse_global")
_ref_rmse  = next((c for c in ("historical", "hist") if c in _rmse_piv.columns), None)
if _ref_rmse is not None:
    _ssp_rmse = [c for c in _rmse_piv.columns if c != _ref_rmse]
    _ratio_df2 = _rmse_piv[_ssp_rmse].divide(_rmse_piv[_ref_rmse], axis=0)
    fig3, ax3 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
    for setup_name in _pred_setup_order:
        if setup_name not in _ratio_df2.index:
            continue
        _rr = _ratio_df2.loc[setup_name]
        ax3.plot(_rr.index, _rr.values, linestyle="None",
                 marker=_pred_markers[setup_name], markersize=7,
                 color=_pred_colors[setup_name], label=setup_name)
    ax3.axhline(1.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
    ax3.set_xlabel("Scenario")
    ax3.set_ylabel(f"RMSE ratio vs. {_ref_rmse}\n({VIZ_PRED_VARIABLE})")
    ax3.set_title(f"RMSE(ssp) / RMSE({_ref_rmse})\n{VIZ_PRED_VARIABLE} (prediction)")
    ax3.grid(axis="y", alpha=0.25)
    ax3.legend(handles=[
        Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
        for s in _pred_setup_order if s in _ratio_df2.index
    ], title="Setup")
    plt.xticks(rotation=20, ha="right")
    plt.show()


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────
# Mean global R² (16 variables) by scenario and setup
# ─────────────────────────────────────────────────────────────────────────────────────

from matplotlib.lines import Line2D

prediction_sources = [
    ("CERA",              complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("Baseline simple",   complete_baseline_simple_quality_df),
]

_pred_setup_order = ["CERA", "Baseline no align", "Baseline simple"]
_pred_markers = {"CERA": "o", "Baseline no align": "s", "Baseline simple": "^"}
_pred_colors  = {"CERA": "#2F3B52", "Baseline no align": "#2C7FB8", "Baseline simple": "#CC7B39"}

# ── Mean global R² (16 variables) per setup ───────────────────────────────────────
r2_mean_pred_rows = []
for setup_name, df in prediction_sources:
    pred_df = df[df["component"] == "prediction"]
    if pred_df.empty:
        continue
    for _, row in pred_df.iterrows():
        r2_dict = row["r2_global"]
        if isinstance(r2_dict, dict) and len(r2_dict) > 0:
            r2_values = [v for k, v in r2_dict.items() if k != "pr"]
            r2_mean = float(np.nanmean(r2_values)) if r2_values else np.nan
        else:
            r2_mean = np.nan

        r2_mean_pred_rows.append({"setup": setup_name, "scenario": row["scenario"], "r2_global_mean": r2_mean})

r2_mean_pred_df = pd.DataFrame(r2_mean_pred_rows)


scenario_order_mean = [c for c in climate_order if c in r2_mean_pred_df["scenario"].unique()]

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in _pred_setup_order:
    sub = r2_mean_pred_df[r2_mean_pred_df["setup"] == setup_name].copy()
    if sub.empty:
        continue
    sub["scenario"] = pd.Categorical(sub["scenario"], categories=scenario_order_mean, ordered=True)
    sub = sub.sort_values("scenario")
    ax.plot(sub["scenario"], sub["r2_global_mean"], linestyle="None",
            marker=_pred_markers[setup_name], markersize=7,
            color=_pred_colors[setup_name], label=setup_name)

ax.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.8)
ax.set_ylim(0.8, 1.1)
ax.set_xlabel("Scenario")
ax.set_ylabel("Mean global $R^2$ (16 variables)")
ax.set_title("Mean global $R^2$ by scenario and setup\n(averaged over 16 reconstructed variables)")
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=[
    Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
    for s in _pred_setup_order
], title="Setup")
plt.xticks(rotation=20, ha="right")
plt.show()

# ── Delta R² vs historical ──────────────────────────────────────────────────────
_r2u_mean = r2_mean_pred_df.groupby(["setup", "scenario"], sort=False)["r2_global_mean"].first().reset_index()
_r2p_mean = _r2u_mean.pivot(index="setup", columns="scenario", values="r2_global_mean")
_ref_mean = next((c for c in ("historical", "hist") if c in _r2p_mean.columns), None)
if _ref_mean is not None:
    _ssps_mean = [c for c in scenario_order_mean if c != _ref_mean and c in _r2p_mean.columns]
    _delta_mean = _r2p_mean[_ssps_mean].subtract(_r2p_mean[_ref_mean], axis=0)
    fig2, ax2 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
    for setup_name in _pred_setup_order:
        if setup_name not in _delta_mean.index:
            continue
        _rd_mean = _delta_mean.loc[setup_name]
        ax2.plot(_rd_mean.index, _rd_mean.values, linestyle="None",
                 marker=_pred_markers[setup_name], markersize=7,
                 color=_pred_colors[setup_name], label=setup_name)
    ax2.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
    ax2.set_xlabel("Scenario")
    ax2.set_ylabel(f"Δ Mean global $R^2$ (16 var.)")
    ax2.set_title(f"Δ Mean global $R^2$ vs. {_ref_mean}\n(averaged over 16 reconstructed variables)")
    ax2.grid(axis="y", alpha=0.25)
    ax2.legend(handles=[
        Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
        for s in _pred_setup_order if s in _delta_mean.index
    ], title="Setup")
    plt.xticks(rotation=20, ha="right")
    plt.show()


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_EXTREME_VARIABLE = "pr"   # variable to visualize
EXTREME_QUANTILE      = 0.90     # percentile threshold for extreme values (e.g., 0.90 for Q90)
# ─────────────────────────────────────────────────────────────────────────────────────

prediction_sources = [
    ("CERA",              complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("Baseline simple",   complete_baseline_simple_quality_df),
]

_pred_setup_order = ["CERA", "Baseline no align", "Baseline simple"]
_pred_markers = {"CERA": "o", "Baseline no align": "s", "Baseline simple": "^"}
_pred_colors  = {"CERA": "#2F3B52", "Baseline no align": "#2C7FB8", "Baseline simple": "#CC7B39"}


def _extreme_rmse(df, scenario, var_name, quantile=EXTREME_QUANTILE):
    """RMSE calculé uniquement sur les points masqués dont la ground truth dépasse
    le quantile `quantile` (ex. q90) de la ground truth pour ce (scenario, variable)."""
    subset = df[(df["component"] == "prediction") & (df["scenario"] == scenario)]
    if subset.empty:
        return np.nan

    t_chunks, p_chunks = [], []
    for _, row in subset.iterrows():
        vnames   = list(row["value_names"])
        col_mask = np.array([str(n).split("@")[0] == var_name for n in vnames])
        t = np.asarray(row["truth_values"], dtype=float)[col_mask]
        p = np.asarray(row["pred_values"],  dtype=float)[col_mask]
        valid = np.isfinite(t) & np.isfinite(p)
        t_chunks.append(t[valid])
        p_chunks.append(p[valid])

    if not t_chunks:
        return np.nan
    t_all = np.concatenate(t_chunks)
    p_all = np.concatenate(p_chunks)
    if t_all.size == 0:
        return np.nan

    threshold = np.quantile(t_all, quantile)
    extreme_mask = t_all >= threshold
    if extreme_mask.sum() < 2:
        return np.nan
    return _safe_rmse(t_all[extreme_mask], p_all[extreme_mask])



# ── RMSE on extreme values (> Q90) per scenario ────────────────────────────────────
rmse_extreme_rows = []
for setup_name, df in prediction_sources:
    scenarios_present = df.loc[df["component"] == "prediction", "scenario"].unique().tolist()
    for scenario in scenarios_present:
        rmse_val = _extreme_rmse(df, scenario, VIZ_EXTREME_VARIABLE)
        rmse_extreme_rows.append({"setup": setup_name, "scenario": scenario, "rmse_extreme": rmse_val})

rmse_extreme_df = pd.DataFrame(rmse_extreme_rows)
scenario_order_extreme = [c for c in climate_order if c in rmse_extreme_df["scenario"].unique()]

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in _pred_setup_order:
    sub = rmse_extreme_df[rmse_extreme_df["setup"] == setup_name].copy()
    if sub.empty:
        continue
    sub["scenario"] = pd.Categorical(sub["scenario"], categories=scenario_order_extreme, ordered=True)
    sub = sub.sort_values("scenario")
    ax.plot(sub["scenario"], sub["rmse_extreme"], linestyle="None",
            marker=_pred_markers[setup_name], markersize=7,
            color=_pred_colors[setup_name], label=setup_name)

ax.set_xlabel("Scenario")
ax.set_ylabel(f"RMSE on extremes (>Q{int(EXTREME_QUANTILE*100)}) — {VIZ_EXTREME_VARIABLE}")
ax.set_title(f"RMSE on extreme ground-truth values (>Q{int(EXTREME_QUANTILE*100)})\n{VIZ_EXTREME_VARIABLE} (prediction)")
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=[
    Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
    for s in _pred_setup_order
], title="Setup")
plt.xticks(rotation=20, ha="right")
plt.show()

# ── Ratio RMSE (extremes) = RMSE(ssp) / RMSE(historical) ─────────────────────────────
_rmse_ext_u = rmse_extreme_df.groupby(["setup", "scenario"], sort=False)["rmse_extreme"].first().reset_index()
_rmse_ext_p = _rmse_ext_u.pivot(index="setup", columns="scenario", values="rmse_extreme")
_ref_ext = next((c for c in ("historical", "hist") if c in _rmse_ext_p.columns), None)
if _ref_ext is not None:
    _ssps_ext = [c for c in scenario_order_extreme if c != _ref_ext and c in _rmse_ext_p.columns]
    _ratio_ext = _rmse_ext_p[_ssps_ext].divide(_rmse_ext_p[_ref_ext], axis=0)
    fig2, ax2 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
    for setup_name in _pred_setup_order:
        if setup_name not in _ratio_ext.index:
            continue
        _rr_ext = _ratio_ext.loc[setup_name]
        ax2.plot(_rr_ext.index, _rr_ext.values, linestyle="None",
                 marker=_pred_markers[setup_name], markersize=7,
                 color=_pred_colors[setup_name], label=setup_name)
    ax2.axhline(1.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
    ax2.set_xlabel("Scenario")
    ax2.set_ylabel(f"RMSE ratio vs. {_ref_ext}\n(extreme >Q{int(EXTREME_QUANTILE*100)}, {VIZ_EXTREME_VARIABLE})")
    ax2.set_title(f"RMSE(ssp) / RMSE({_ref_ext}) — extremes\n{VIZ_EXTREME_VARIABLE} (prediction)")
    ax2.grid(axis="y", alpha=0.25)
    ax2.legend(handles=[
        Line2D([0], [0], color=_pred_colors[s], marker=_pred_markers[s], linewidth=0.0, label=s)
        for s in _pred_setup_order if s in _ratio_ext.index
    ], title="Setup")
    plt.xticks(rotation=20, ha="right")
    plt.show()


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
HEATMAP_REF_CLIMATE    = "historical"
HEATMAP_TARGET_CLIMATE = "ssp585"
# ─────────────────────────────────────────────────────────────────────────────────────

prediction_sources = [
    ("CERA",              complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("Baseline simple",   complete_baseline_simple_quality_df),
]
_pred_setup_dict  = dict(prediction_sources)
_pred_setup_order = ["CERA", "Baseline no align", "Baseline simple"]


def _get_r2_dict(df, scenario):
    """Récupère le dict {variable: r2} pour le composant 'prediction' d'un scénario donné."""
    sub = df[(df["component"] == "prediction") & (df["scenario"] == scenario)]
    if sub.empty:
        return None
    r2 = sub.iloc[0]["r2_global"]
    return r2 if isinstance(r2, dict) else None

variable_order_hm = [v for v in selected_variables_full if v != "pr"]

delta_matrix = np.full((len(_pred_setup_order), len(variable_order_hm)), np.nan)

for si, setup_name in enumerate(_pred_setup_order):
    r2_ref = _get_r2_dict(_pred_setup_dict[setup_name], HEATMAP_REF_CLIMATE)
    r2_tgt = _get_r2_dict(_pred_setup_dict[setup_name], HEATMAP_TARGET_CLIMATE)
    if r2_ref is None or r2_tgt is None:
        continue
    for vi, var in enumerate(variable_order_hm):
        if var in r2_ref and var in r2_tgt:
            delta_matrix[si, vi] = r2_tgt[var] - r2_ref[var]

fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(variable_order_hm)), 3.5), constrained_layout=True)
vmax = np.nanmax(np.abs(delta_matrix)) if np.isfinite(delta_matrix).any() else 1.0
im = ax.imshow(delta_matrix, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

ax.set_xticks(range(len(variable_order_hm)))
ax.set_xticklabels(variable_order_hm, rotation=45, ha="right")
ax.set_yticks(range(len(_pred_setup_order)))
ax.set_yticklabels(_pred_setup_order)

for si in range(len(_pred_setup_order)):
    for vi in range(len(variable_order_hm)):
        val = delta_matrix[si, vi]
        if np.isfinite(val):
            ax.text(vi, si, f"{val:.2f}", ha="center", va="center",
                    color="white" if abs(val) > vmax * 0.5 else "black", fontsize=7)

ax.set_xlabel("Variable")
ax.set_ylabel("Setup")
ax.set_title(f"Δ Global $R^2$ ({HEATMAP_TARGET_CLIMATE} − {HEATMAP_REF_CLIMATE})\nby variable and setup (prediction)")
fig.colorbar(im, ax=ax, label=f"Δ $R^2$ ({HEATMAP_TARGET_CLIMATE} − {HEATMAP_REF_CLIMATE})")
plt.show()


In [ ]:
# Distribution of R² values by setup and climate scenario
_climate_order_v = climate_order
_setup_order_v = ["CERA", "Baseline no align", "Baseline simple"]
_violin_colors  = {"CERA": "#2F3B52", "Baseline no align": "#2C7FB8", "Baseline simple": "#CC7B39"}

_samples_v = {c: {s: []     for s in _setup_order_v} for c in _climate_order_v}
_r2g_v     = {c: {s: np.nan for s in _setup_order_v} for c in _climate_order_v}

for _sname, _sdf in prediction_sources:
    _pred_v = _sdf[_sdf["component"] == "prediction"]
    if _pred_v.empty or "r2" not in _pred_v.columns:
        continue

    for _clim_raw, _grp in _pred_v.groupby("scenario", sort=False):
        _clim = str(_clim_raw)
        if _clim not in _climate_order_v:
            continue

        if "r2_global" in _grp.columns:
            _rg = _grp.iloc[0]["r2_global"]
            if isinstance(_rg, dict):
                _rg = _rg.get(VIZ_PRED_VARIABLE, np.nan)
            if np.isfinite(float(_rg)):
                _r2g_v[_clim][_sname] = float(_rg)

        for _, _row in _grp.iterrows():
            _rs = _row["r2"]
            if isinstance(_rs, dict):
                _rs = _rs.get(VIZ_PRED_VARIABLE, np.nan)
            if np.isfinite(float(_rs)):
                _samples_v[_clim][_sname].append(float(_rs))

fig, axes = plt.subplots(1, 4, figsize=(16, 5), constrained_layout=True, sharey=True)

for _ci, _clim in enumerate(_climate_order_v):
    ax = axes[_ci]
    _pos, _data, _globals, _labels, _cols = [], [], [], [], []
    for _i, _sname in enumerate(_setup_order_v):
        _s = _samples_v[_clim][_sname]
        if not _s:
            continue
        _pos.append(_i + 1)
        _data.append(_s)
        _globals.append(_r2g_v[_clim][_sname])
        _labels.append(_sname)
        _cols.append(_violin_colors[_sname])

    if not _data:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, color="grey")
        ax.set_title(_clim.upper(), fontweight="bold")
        continue

    _vp = ax.violinplot(_data, positions=_pos, showmedians=True, showextrema=True)
    for _vi, _body in enumerate(_vp["bodies"]):
        _body.set_facecolor(_cols[_vi])
        _body.set_alpha(0.6)
        _body.set_edgecolor(_cols[_vi])
    for _part in ("cmins", "cmaxes", "cbars"):
        if _part in _vp:
            _vp[_part].set_color("grey")
            _vp[_part].set_linewidth(0.8)
    _vp["cmedians"].set_color("white")
    _vp["cmedians"].set_linewidth(2.0)
    _vp["cmedians"].set_zorder(4)

    for _vi, (_p, _rg) in enumerate(zip(_pos, _globals)):
        if np.isfinite(_rg):
            ax.scatter([_p], [_rg], marker="D", s=55, color="red", zorder=5,
                       label="$R^2$ global" if (_ci == 0 and _vi == 0) else "_nolegend_")

    ax.axhline(0.0, color="#6B7280", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_xticks(_pos)
    ax.set_xticklabels(_labels, rotation=20, ha="right", fontsize=9)
    ax.set_title(_clim.upper(), fontweight="bold")
    ax.grid(axis="y", alpha=0.25, linestyle=":")

axes[0].set_ylabel(f"$R^2$ ({VIZ_PRED_VARIABLE})", fontweight="bold")
axes[-1].legend(
    handles=[Line2D([0], [0], marker="D", color="red", linewidth=0, markersize=6, label="$R^2$ global")],
    loc="lower right", fontsize=9,
)
fig.suptitle(
    f"Distribution du $R^2$ de prédiction par setup et par climat  ·  {VIZ_PRED_VARIABLE}",
    fontweight="bold", fontsize=13,
)
plt.show()

In [ ]:
MAX_SCATTER_PTS = 25000

def _extract_mask_pred_pairs(df, climate_name, var_name, max_pts=MAX_SCATTER_PTS, seed=0):
    # One point per sample: mean over all masked points of var_name (like variable notebook)
    subset = df[(df["component"] == "prediction") & (df["scenario"] == climate_name)]
    if subset.empty or "truth_values" not in subset.columns:
        return np.array([]), np.array([])

    t_means, p_means = [], []
    for _, row in subset.iterrows():
        vnames   = list(row["value_names"])
        t_flat   = np.asarray(row["truth_values"], dtype=float)
        p_flat   = np.asarray(row["pred_values"],  dtype=float)
        col_mask = np.array([str(n).split("@")[0] == var_name for n in vnames])
        t_var    = t_flat[col_mask]
        p_var    = p_flat[col_mask]
        valid    = np.isfinite(t_var) & np.isfinite(p_var)
        if valid.any():
            t_means.append(float(np.nanmean(t_var[valid])))
            p_means.append(float(np.nanmean(p_var[valid])))

    t_arr = np.asarray(t_means, dtype=float)
    p_arr = np.asarray(p_means, dtype=float)
    if t_arr.size > max_pts:
        rng = np.random.default_rng(seed)
        idx = rng.choice(t_arr.size, size=max_pts, replace=False)
        t_arr, p_arr = t_arr[idx], p_arr[idx]
    return t_arr, p_arr

climates_pred = complete_cera_quality_df["scenario"].unique().tolist()
setups_scatter = [
    (complete_cera_quality_df,                    "CERA"),
    (complete_baseline_cera_noalign_quality_df,    "Baseline CERA\n(no align)"),
    (complete_baseline_simple_quality_df,          "Baseline Simple"),
]

all_data_sc = {}
gt_min = gp_min = gz_min =  np.inf
gt_max = gp_max = gz_max = -np.inf

for ci, clim in enumerate(climates_pred):
    for si, (sdf, stitle) in enumerate(setups_scatter):
        t, p = _extract_mask_pred_pairs(sdf, clim, VIZ_PRED_VARIABLE)
        if t.size == 0:
            continue
        gt_min = min(gt_min, t.min()); gt_max = max(gt_max, t.max())
        gp_min = min(gp_min, p.min()); gp_max = max(gp_max, p.max())

        dens = None
        if t.size >= 3 and np.unique(np.column_stack([t, p]), axis=0).shape[0] >= 3:
            try:
                dens = gaussian_kde(np.vstack([t, p]))(np.vstack([t, p]))
                gz_min = min(gz_min, dens.min()); gz_max = max(gz_max, dens.max())
            except Exception:
                dens = None

        pred_sub = sdf[(sdf["component"] == "prediction") & (sdf["scenario"] == clim)]
        rmse_g = np.nan
        if not pred_sub.empty and "rmse_global" in pred_sub.columns:
            _rv = pred_sub.iloc[0]["rmse_global"]
            if isinstance(_rv, dict):
                rmse_g = _rv.get(VIZ_PRED_VARIABLE, np.nan)
            else:
                rmse_g = _rv
        all_data_sc[(ci, si)] = {"t": t, "p": p, "dens": dens, "rmse": rmse_g}

if not np.isfinite(gt_min):
    raise ValueError("No finite truth values for scatter plot.")

xpad = 0.05 * (gt_max - gt_min) if gt_max > gt_min else 1.0
ypad = 0.05 * (gp_max - gp_min) if gp_max > gp_min else 1.0
gt_min -= xpad; gt_max += xpad; gp_min -= ypad; gp_max += ypad

dens_norm = Normalize(vmin=gz_min, vmax=gz_max) if np.isfinite(gz_min) else None

fig, axes = plt.subplots(len(climates_pred), 3,
                         figsize=(18, 4 * len(climates_pred)), constrained_layout=False)
if len(climates_pred) == 1:
    axes = axes[np.newaxis, :]

for ci, clim in enumerate(climates_pred):
    for si, (_, stitle) in enumerate(setups_scatter):
        ax = axes[ci, si]
        d  = all_data_sc.get((ci, si))
        if d is None or d["t"].size == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=10)
        else:
            if d["dens"] is not None and dens_norm is not None:
                ax.scatter(d["t"], d["p"], c=d["dens"], s=12, cmap="hot",
                           alpha=0.65, edgecolors="none", norm=dens_norm)
            else:
                ax.scatter(d["t"], d["p"], s=12, c="#6B7280", alpha=0.55, edgecolors="none")
            ax.plot([gt_min, gt_max], [gt_min, gt_max], "k--", linewidth=1.2, alpha=0.7)
            rmse_txt = f"{d['rmse']:.5f}" if np.isfinite(d["rmse"]) else "N/A"
            ax.set_title(f"{stitle}\nRMSE global = {rmse_txt}", fontsize=11, fontweight="bold")

        ax.set_xlim(gt_min, gt_max)
        ax.set_ylim(gp_min, gp_max)
        ax.set_xlabel("True value")
        ax.set_ylabel("Predicted value")
        ax.grid(alpha=0.25, linestyle=":")

    axes[ci, 0].text(-0.35, 0.5, clim.upper(), transform=axes[ci, 0].transAxes,
                     fontsize=13, fontweight="bold", ha="center", va="center", rotation=90)

if dens_norm is not None:
    plt.subplots_adjust(left=0.1, right=0.88, top=0.93, bottom=0.08, wspace=0.3, hspace=0.38)
    cax = fig.add_axes([0.90, 0.15, 0.018, 0.7])
    fig.colorbar(plt.cm.ScalarMappable(norm=dens_norm, cmap="hot"), cax=cax).set_label("Density", fontsize=11)
else:
    plt.subplots_adjust(left=0.1, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.38)

plt.suptitle(
    f"Prediction scatter  ·  {VIZ_PRED_VARIABLE}  ·  True vs Predicted (masked points)",
    fontsize=15, fontweight="bold", y=0.995,
)
plt.show()


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_LAT_P = "historical"  
N_LAT_BINS_P      = 30
# ─────────────────────────────────────────────────────────────────────────────────────


if "_patch_catalog" not in dir():
    for _n in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
        _pp = precomputed_dir / _n
        if _pp.exists():
            with open(_pp, "rb") as _f:
                _patch_catalog = pickle.load(_f)
            break
_cat_lp = _patch_catalog.set_index("patch_id")

_vis_idx_p  = np.asarray(mask_constants.get("visible_point_indices", []), dtype=int)
_mask_idx_p = np.asarray(mask_constants.get("masked_point_indices",  []), dtype=int)

_pred_sources_lp = [
    ("CERA",              complete_cera_quality_df,                 "#2F3B52"),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df,"#2C7FB8"),
    ("Baseline simple",   complete_baseline_simple_quality_df,      "#CC7B39"),
]

_lat_r2_pred = {}
for _sname, _sdf, _col in _pred_sources_lp:
    _filt = _sdf[
        (_sdf["component"] == "prediction") &
        (_sdf["scenario"] == VIZ_CLIMATE_LAT_P)
    ]
    if _filt.empty or "patch_id" not in _filt.columns or "r2" not in _filt.columns:
        continue

    _pairs_lp = []
    for _, _row in _filt.iterrows():
        _pid = int(_row["patch_id"])
        if _pid not in _cat_lp.index:
            continue
        _info_lp = _cat_lp.loc[_pid]
        _lat_c   = (float(_info_lp["lat_start"]) + float(_info_lp["lat_stop"])) / 2.0
        _r2_dict = _row["r2"]
        if isinstance(_r2_dict, dict):
            _r2_val = _r2_dict.get(VIZ_PRED_VARIABLE, np.nan)
            if np.isfinite(float(_r2_val)):
                _pairs_lp.append((_lat_c, float(_r2_val)))
    if _pairs_lp:
        _lat_r2_pred[_sname] = (_pairs_lp, _col)

if not _lat_r2_pred:
    raise ValueError("No valid R² data found for the specified climate scenario and variable.")

_all_lats_lp = [lat for _sl, _ in _lat_r2_pred.values() for lat, _ in _sl]
_bins_lp     = np.linspace(min(_all_lats_lp), max(_all_lats_lp), N_LAT_BINS_P + 1)
_ctrs_lp     = (_bins_lp[:-1] + _bins_lp[1:]) / 2

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
for _ci_lp, (_sname, (_pairs_lp, _col)) in enumerate(_lat_r2_pred.items()):
    _la = np.array([p[0] for p in _pairs_lp])
    _ra = np.array([p[1] for p in _pairs_lp])
    _mr = np.full(N_LAT_BINS_P, np.nan)
    _sr = np.full(N_LAT_BINS_P, np.nan)
    for _bi in range(N_LAT_BINS_P):
        _m = (_la >= _bins_lp[_bi]) & (_la < _bins_lp[_bi + 1])
        _n = int(np.sum(_m))
        if _n >= 2:
            _mr[_bi] = np.mean(_ra[_m])
            _sr[_bi] = np.std(_ra[_m], ddof=1)
        elif _n == 1:
            _mr[_bi] = _ra[_m][0]
    _v = np.isfinite(_mr)
    _sc_ = np.where(np.isfinite(_sr[_v]), _sr[_v], 0.0)
    ax.plot(_ctrs_lp[_v], _mr[_v], linewidth=1.8, color=_col, label=_sname)
    ax.fill_between(_ctrs_lp[_v], _mr[_v] - _sc_, _mr[_v] + _sc_, alpha=0.15, color=_col)

ax.axhline(0.0, color="#6B7280", linewidth=0.9, linestyle="--", alpha=0.7)
ax.set_xlabel("Latitude (°)", fontweight="bold")
ax.set_ylabel(f"$R^2$ ({VIZ_PRED_VARIABLE})", fontweight="bold")
ax.set_title(
    f"$R^2$ of prediction by latitude  ·  {VIZ_PRED_VARIABLE}  ·  {VIZ_CLIMATE_LAT_P}\n"
    "(mean ± standard deviation, latitude = center of the patch)",
    fontweight="bold",
)
ax.legend(title="Setup", fontsize=9, loc="best", framealpha=0.9)
ax.grid(alpha=0.25, linestyle=":")
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP_CONF_P   = "CERA"    # "CERA", "Baseline no align", "Baseline simple"
VIZ_CLIMATE_CONF_P = "historical"   
N_BINS_CONF_P      = 20
NORMALIZE_CONF_P   = True
# ─────────────────────────────────────────────────────────────────────────────────────

_conf_srcs_p = {
    "CERA": complete_cera_quality_df,
    "Baseline no align": complete_baseline_cera_noalign_quality_df,
    "Baseline simple":   complete_baseline_simple_quality_df,
}

_df_cp = _conf_srcs_p[VIZ_SETUP_CONF_P]
_pred_cp = _df_cp[
    (_df_cp["component"] == "prediction") &
    (_df_cp["scenario"] == VIZ_CLIMATE_CONF_P)
]
if _pred_cp.empty:
    raise ValueError(f"No valid data found for setup='{VIZ_SETUP_CONF_P}', climate='{VIZ_CLIMATE_CONF_P}'.")

_all_t_cp, _all_p_cp = [], []
for _, _row in _pred_cp.iterrows():
    _vn = list(_row["value_names"])
    _cm = np.array([str(n).split("@")[0] == VIZ_PRED_VARIABLE for n in _vn])
    _t  = np.asarray(_row["truth_values"], dtype=float)[_cm]
    _p  = np.asarray(_row["pred_values"],  dtype=float)[_cm]
    _mf = np.isfinite(_t) & np.isfinite(_p)
    _all_t_cp.extend(_t[_mf].tolist())
    _all_p_cp.extend(_p[_mf].tolist())

_all_t_cp = np.array(_all_t_cp)
_all_p_cp = np.array(_all_p_cp)
if len(_all_t_cp) == 0:
    raise ValueError("No valid finite values available for the confusion matrix.")

_bin_edges_p = np.unique(np.percentile(_all_t_cp, np.linspace(0, 100, N_BINS_CONF_P + 1)))
_nb_p = len(_bin_edges_p) - 1
if _nb_p < 2:
    raise ValueError("Not enough distinct values for the bins. Reduce N_BINS_CONF_P.")

_truth_idx_p = np.clip(np.digitize(_all_t_cp, _bin_edges_p[1:-1]), 0, _nb_p - 1)
_pred_idx_p  = np.digitize(_all_p_cp, _bin_edges_p)
_nc_p        = _nb_p + 2

_conf_mat_p = np.zeros((_nb_p, _nc_p), dtype=float)
np.add.at(_conf_mat_p, (_truth_idx_p, _pred_idx_p), 1)

if NORMALIZE_CONF_P:
    _rs = _conf_mat_p.sum(axis=1, keepdims=True)
    _rs[_rs == 0] = 1
    _conf_plot_p = _conf_mat_p / _rs
    _cbar_lbl_p  = "Fraction (normalized by row)"
    _afmt_p      = ".2f"
else:
    _conf_plot_p = _conf_mat_p
    _cbar_lbl_p  = "Number of points"
    _afmt_p      = ".0f"

_bin_lbl_p = [f"[{_bin_edges_p[i]:.2g}, {_bin_edges_p[i+1]:.2g})" for i in range(_nb_p)]
_col_lbl_p = ([f"< {_bin_edges_p[0]:.2g}"] + _bin_lbl_p + [f"≥ {_bin_edges_p[-1]:.2g}"])

fig, ax = plt.subplots(figsize=(max(8, 7), 7), constrained_layout=True)
_im_cp = ax.imshow(_conf_plot_p, aspect="auto", cmap="Blues", origin="lower",
                   vmin=0, vmax=_conf_plot_p.max())
for _i in range(_nb_p):
    for _j in range(_nc_p):
        _val = _conf_plot_p[_i, _j]
        ax.text(_j, _i, f"{_val:{_afmt_p}}",
                ha="center", va="center", fontsize=7,
                color="white" if _val > 0.55 * _conf_plot_p.max() else "black")
for _k in range(_nb_p):
    ax.add_patch(plt.Rectangle((_k + 0.5, _k - 0.5), 1, 1,
                                fill=False, edgecolor="#E63946", linewidth=1.2))
ax.axvline(0.5,           color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)
ax.axvline(_nc_p - 1 - 0.5, color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)
ax.set_xticks(range(_nc_p))
ax.set_xticklabels(_col_lbl_p, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(_nb_p))
ax.set_yticklabels(_bin_lbl_p, fontsize=8)
ax.set_xlabel("Bin predicted", fontweight="bold")
ax.set_ylabel("Bin true (quantile)", fontweight="bold")
ax.set_title(
    f"Confusion matrix  ·  {VIZ_SETUP_CONF_P}  ·  {VIZ_CLIMATE_CONF_P}  ·  {VIZ_PRED_VARIABLE}\n"
    f"({'normalized by row' if NORMALIZE_CONF_P else 'raw count'}"
    f", {len(_all_t_cp):,} points, {_nb_p} quantile bins)",
    fontweight="bold",
)
fig.colorbar(_im_cp, ax=ax, label=_cbar_lbl_p, fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP_PRED    = "CERA"    # "CERA", "Baseline no align", "Baseline simple"
VIZ_CLIMATE_PRED  = "historical"    
VIZ_SAMPLE_IDX_P  = 0
# ─────────────────────────────────────────────────────────────────────────────────────

if "_patch_catalog" not in dir():
    for _n in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
        _pp = precomputed_dir / _n
        if _pp.exists():
            with open(_pp, "rb") as _f:
                _patch_catalog = pickle.load(_f)
            break

_pred_srcs_gp = {
    "CERA": complete_cera_quality_df,
    "Baseline no align": complete_baseline_cera_noalign_quality_df,
    "Baseline simple":   complete_baseline_simple_quality_df,
}
_baseline_names_gp = ["Baseline no align", "Baseline simple"]

_src_gp = _pred_srcs_gp[VIZ_SETUP_PRED]
_main_gp = _src_gp[
    (_src_gp["component"] == "prediction") &
    (_src_gp["scenario"] == VIZ_CLIMATE_PRED)
].reset_index(drop=True)

if _main_gp.empty:
    raise ValueError(f"No data for setup='{VIZ_SETUP_PRED}', climat='{VIZ_CLIMATE_PRED}'.")
if VIZ_SAMPLE_IDX_P >= len(_main_gp):
    raise IndexError(f"VIZ_SAMPLE_IDX_P={VIZ_SAMPLE_IDX_P} hors limites (max {len(_main_gp)-1}).")

_row_gp   = _main_gp.iloc[VIZ_SAMPLE_IDX_P]
_patch_gp = int(_row_gp["patch_id"])
_info_gp  = _patch_catalog.loc[_patch_catalog["patch_id"] == _patch_gp].iloc[0]
_n_lat_gp = int(_info_gp["lat_stop_idx"] - _info_gp["lat_start_idx"])
_n_lon_gp = int(_info_gp["lon_stop_idx"] - _info_gp["lon_start_idx"])
_lats_gp  = np.linspace(float(_info_gp["lat_start"]), float(_info_gp["lat_stop"]), _n_lat_gp)
_lons_gp  = np.linspace(float(_info_gp["lon_start"]), float(_info_gp["lon_stop"]), _n_lon_gp)

_mask_idx_gp = np.asarray(mask_constants.get("masked_point_indices",  []), dtype=int)
_vis_idx_gp  = np.asarray(mask_constants.get("visible_point_indices",  []), dtype=int)
_mk_rows = _mask_idx_gp // _n_lon_gp; _mk_cols = _mask_idx_gp % _n_lon_gp
_vk_rows = _vis_idx_gp  // _n_lon_gp; _vk_cols = _vis_idx_gp  % _n_lon_gp
_mask_lats_gp = _lats_gp[_mk_rows]; _mask_lons_gp = _lons_gp[_mk_cols]
_vis_lats_gp  = _lats_gp[_vk_rows]; _vis_lons_gp  = _lons_gp[_vk_cols]

# Extracted truth and predicted values for the selected variable
_vn_gp   = list(_row_gp["value_names"])
_cm_gp   = np.array([str(n).split("@")[0] == VIZ_PRED_VARIABLE for n in _vn_gp])
_truth_gp = np.asarray(_row_gp["truth_values"], dtype=float)[_cm_gp]
_pred_gp  = np.asarray(_row_gp["pred_values"],  dtype=float)[_cm_gp]

# Panels: [main setup pred] + [baselines] (matched by patch_id)
_panels_gp = [(VIZ_SETUP_PRED, _pred_gp)]
for _bn in _baseline_names_gp:
    _bdf = _pred_srcs_gp[_bn]
    _bf  = _bdf[
        (_bdf["component"] == "prediction") &
        (_bdf["scenario"] == VIZ_CLIMATE_PRED) &
        (_bdf["patch_id"] == _patch_gp)
    ].reset_index(drop=True)
    if _bf.empty:
        print(f"[WARN] No sample for {_bn} / patch_id={_patch_gp} / {VIZ_CLIMATE_PRED} — skipped.")
        _panels_gp.append((_bn, None))
    else:
        _vn_b  = list(_bf.iloc[0]["value_names"])
        _cm_b  = np.array([str(n).split("@")[0] == VIZ_PRED_VARIABLE for n in _vn_b])
        _panels_gp.append((_bn, np.asarray(_bf.iloc[0]["pred_values"], dtype=float)[_cm_b]))

_all_arrs_gp = [_truth_gp] + [a for _, a in _panels_gp if a is not None]
_vmin_gp = float(min(np.nanmin(a) for a in _all_arrs_gp))
_vmax_gp = float(max(np.nanmax(a) for a in _all_arrs_gp))
_norm_gp = plt.Normalize(vmin=_vmin_gp, vmax=_vmax_gp)
_cmap_gp = "Blues"
_proj_gp = ccrs.PlateCarree()

_lon_min_gp = _lons_gp.min(); _lon_max_gp = _lons_gp.max()
_lat_min_gp = _lats_gp.min(); _lat_max_gp = _lats_gp.max()
_mg_lo = max(1.0, 0.2 * (_lon_max_gp - _lon_min_gp))
_mg_la = max(1.0, 0.2 * (_lat_max_gp - _lat_min_gp))
_ext_gp = [_lon_min_gp - _mg_lo, _lon_max_gp + _mg_lo,
           _lat_min_gp - _mg_la, _lat_max_gp + _mg_la]

_np_gp = len(_panels_gp)
fig = plt.figure(figsize=(4.5 * (_np_gp + 1) + 2.5, 5.5), constrained_layout=True)
gs  = gridspec.GridSpec(1, _np_gp + 2, figure=fig,
                        width_ratios=[2] * (_np_gp + 1) + [1.4])


def _map_p_gp(pos):
    ax = fig.add_subplot(pos, projection=_proj_gp)
    ax.set_extent(_ext_gp, crs=_proj_gp)
    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.35)
    gl.top_labels = gl.right_labels = False
    return ax


import numpy.ma as _nma_p
_cmap_gp_obj = plt.get_cmap(_cmap_gp).copy()
_cmap_gp_obj.set_bad(alpha=0)

_im_gp = None
_col_axes_gp = []
for _col, (_lbl, _parr) in enumerate(_panels_gp):
    _ax = _map_p_gp(gs[_col])
    _col_axes_gp.append(_ax)
    # Visible points as gray X
    _ax.scatter(_vis_lons_gp, _vis_lats_gp, s=40, color="gray",
                marker="x", linewidths=1.2, transform=_proj_gp, zorder=3)
    if _parr is not None:
        # pcolormesh: fill only masked positions
        _pred_2d_gp = np.full((_n_lat_gp, _n_lon_gp), np.nan)
        _pred_2d_gp[_mk_rows, _mk_cols] = _parr
        _im_gp = _ax.pcolormesh(
            np.meshgrid(np.linspace(float(_lons_gp[0]), float(_lons_gp[-1]), _n_lon_gp),
                        np.linspace(float(_lats_gp[0]), float(_lats_gp[-1]), _n_lat_gp))[0],
            np.meshgrid(np.linspace(float(_lons_gp[0]), float(_lons_gp[-1]), _n_lon_gp),
                        np.linspace(float(_lats_gp[0]), float(_lats_gp[-1]), _n_lat_gp))[1],
            _nma_p.masked_invalid(_pred_2d_gp),
            shading="auto", cmap=_cmap_gp_obj, norm=_norm_gp, transform=_proj_gp, zorder=4,
        )
    else:
        _ax.text(0.5, 0.5, "N/A", transform=_ax.transAxes,
                 ha="center", va="center", fontsize=14, color="gray")
    _ax.set_title(f"Prediction  ({_lbl})\n{VIZ_PRED_VARIABLE}  ·  {VIZ_CLIMATE_PRED}  ·  patch {_patch_gp}",
                  fontweight="bold")

_ax_truth_gp = _map_p_gp(gs[_np_gp])
_col_axes_gp.append(_ax_truth_gp)
# Ground truth
_truth_2d_gp = np.full((_n_lat_gp, _n_lon_gp), np.nan)
_truth_2d_gp[_mk_rows, _mk_cols] = _truth_gp
_lon_grid_gp, _lat_grid_gp = np.meshgrid(
    np.linspace(float(_lons_gp[0]), float(_lons_gp[-1]), _n_lon_gp),
    np.linspace(float(_lats_gp[0]), float(_lats_gp[-1]), _n_lat_gp),
)
_ax_truth_gp.scatter(_vis_lons_gp, _vis_lats_gp, s=40, color="gray",
                     marker="x", linewidths=1.2, transform=_proj_gp, zorder=3)
_ax_truth_gp.pcolormesh(
    _lon_grid_gp, _lat_grid_gp, _nma_p.masked_invalid(_truth_2d_gp),
    shading="auto", cmap=_cmap_gp_obj, norm=_norm_gp, transform=_proj_gp, zorder=4,
)
_ax_truth_gp.set_title(f"Ground Truth\n{VIZ_PRED_VARIABLE}  ·  {VIZ_CLIMATE_PRED}  ·  sample #{VIZ_SAMPLE_IDX_P}",
                       fontweight="bold")

if _im_gp is not None:
    fig.colorbar(_im_gp, ax=_col_axes_gp, orientation="vertical",
                 fraction=0.02, pad=0.02, label=VIZ_PRED_VARIABLE)

_ax_world_gp = fig.add_subplot(gs[_np_gp + 1], projection=_proj_gp)
_ax_world_gp.set_global()
_ax_world_gp.coastlines(resolution="110m", linewidth=0.7, color="black")
_ax_world_gp.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
_ax_world_gp.gridlines(linewidth=0.3, alpha=0.35)
_ax_world_gp.add_patch(mpatches.Rectangle(
    (_lon_min_gp, _lat_min_gp), _lon_max_gp - _lon_min_gp, _lat_max_gp - _lat_min_gp,
    linewidth=2.0, edgecolor="red", facecolor="none", transform=_proj_gp, zorder=5,
))
_ax_world_gp.set_title(f"Localisation\npatch_id = {_patch_gp}", fontweight="bold")

fig.suptitle(
    f"Prediction vs Ground Truth (masked points)  ·  {VIZ_CLIMATE_PRED}  ·  {VIZ_PRED_VARIABLE}",
    fontweight="bold", fontsize=13,
)
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_DIST_P = "historical"  
VIZ_N_BINS_DIST_P  = 60
# ─────────────────────────────────────────────────────────────────────────────────────

_pred_dist = {
    "CERA":              complete_cera_quality_df,
    "Baseline no align": complete_baseline_cera_noalign_quality_df,
    "Baseline simple":   complete_baseline_simple_quality_df,
}

_n_d = len(_pred_dist)
fig, axes = plt.subplots(2, _n_d, figsize=(6 * _n_d, 10))

for _col, (_sname, _sdf) in enumerate(_pred_dist.items()):
    ax_hist = axes[0, _col]
    ax_cdf  = axes[1, _col]

    _mask_d = _sdf["component"] == "prediction"
    if VIZ_CLIMATE_DIST_P is not None:
        _mask_d &= _sdf["scenario"] == VIZ_CLIMATE_DIST_P
    _df_d = _sdf[_mask_d]
    if _df_d.empty:
        ax_hist.set_title(f"{_sname}\n(no data for {VIZ_CLIMATE_DIST_P})")
        continue

    _t_all_d, _p_all_d = [], []
    for _, _row in _df_d.iterrows():
        _vn_d = list(_row["value_names"])
        _cm_d = np.array([str(n).split("@")[0] == VIZ_PRED_VARIABLE for n in _vn_d])
        _t_d  = np.asarray(_row["truth_values"], dtype=float)[_cm_d]
        _p_d  = np.asarray(_row["pred_values"],  dtype=float)[_cm_d]
        _mf_d = np.isfinite(_t_d) & np.isfinite(_p_d)
        _t_all_d.extend(_t_d[_mf_d].tolist())
        _p_all_d.extend(_p_d[_mf_d].tolist())

    _t_all_d = np.array(_t_all_d)
    _p_all_d = np.array(_p_all_d)
    if _t_all_d.size == 0:
        ax_hist.set_title(f"{_sname}\n(no finite values)")
        continue

    _xmin_d = min(_t_all_d.min(), _p_all_d.min())
    _xmax_d = max(_t_all_d.max(), _p_all_d.max())
    _bins_d = np.linspace(_xmin_d, _xmax_d, VIZ_N_BINS_DIST_P + 1)
    _xs_d   = np.linspace(_xmin_d, _xmax_d, 400)

    ax_hist.hist(_t_all_d, bins=_bins_d, density=True, alpha=0.3, color="steelblue", label="Ground Truth")
    ax_hist.hist(_p_all_d, bins=_bins_d, density=True, alpha=0.3, color="tomato",    label="Prédiction")
    for _vals_d, _col_d in [(_t_all_d, "steelblue"), (_p_all_d, "tomato")]:
        _kde_d = gaussian_kde(_vals_d, bw_method="silverman")
        ax_hist.plot(_xs_d, _kde_d(_xs_d), color=_col_d, linewidth=2)
        ax_hist.axvline(_vals_d.mean(), color=_col_d, linestyle="--", linewidth=1.2, alpha=0.8)

    _txt_d = (
        f"Truth : μ={_t_all_d.mean():.4g}  σ={_t_all_d.std():.4g}\n"
        f"        p5={np.percentile(_t_all_d, 5):.4g}  p95={np.percentile(_t_all_d, 95):.4g}\n"
        f"Pred  : μ={_p_all_d.mean():.4g}  σ={_p_all_d.std():.4g}\n"
        f"        p5={np.percentile(_p_all_d,  5):.4g}  p95={np.percentile(_p_all_d, 95):.4g}"
    )
    ax_hist.text(0.98, 0.98, _txt_d, transform=ax_hist.transAxes,
                 ha="right", va="top", fontsize=8, family="monospace",
                 bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.8))
    _clim_d = VIZ_CLIMATE_DIST_P or "all"
    ax_hist.set_title(f"{_sname}  ·  {_clim_d}  ·  {VIZ_PRED_VARIABLE}", fontweight="bold")
    ax_hist.set_xlabel(VIZ_PRED_VARIABLE)
    ax_hist.set_ylabel("Densité")
    ax_hist.legend(fontsize=9)

    for _vals_d, _col_d, _lbl_d in [
        (_t_all_d, "steelblue", "Ground Truth"), (_p_all_d, "tomato", "Prediction")
    ]:
        _sorted_d = np.sort(_vals_d)
        _cdf_d    = np.arange(1, len(_sorted_d) + 1) / len(_sorted_d)
        ax_cdf.plot(_sorted_d, _cdf_d, color=_col_d, linewidth=2, label=_lbl_d)
        ax_cdf.axvline(_vals_d.mean(), color=_col_d, linestyle="--", linewidth=1.0, alpha=0.7)
    for _vals_d, _col_d in [(_t_all_d, "steelblue"), (_p_all_d, "tomato")]:
        ax_cdf.axvspan(np.percentile(_vals_d, 25), np.percentile(_vals_d, 75), alpha=0.07, color=_col_d)
    ax_cdf.set_title(f"CDF  ·  {_sname}", fontweight="bold")
    ax_cdf.set_xlabel(VIZ_PRED_VARIABLE)
    ax_cdf.set_ylabel("Cumulative probability")
    ax_cdf.legend(fontsize=9)
    ax_cdf.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax_cdf.grid(True, alpha=0.3)

fig.suptitle(
    f"Statistical distribution of {VIZ_PRED_VARIABLE}  ·  Ground Truth vs Prediction  ·  {VIZ_CLIMATE_DIST_P or 'all climates'}",
    fontweight="bold", fontsize=14,
)
plt.tight_layout()
plt.show()

## Part 4 - Plotting history results

### *CERA*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

colors = {
    "recon": "#2C7FB8",  
    "align": "#7F7F7F",  
    "pred":  "#6A3D9A",   
    "total": "#D62728", 
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "align": ["train_align_loss", "val_align_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        is_train = "train" in col
        
        ax.plot(
            cera_history_df["epoch"],
            cera_history_df[col],
            linestyle="--" if is_train else "-", 
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title("CERA training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")

ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)

ax.legend(title="Loss components", ncol=2, fontsize=9)

plt.show()

### *CERA noAlign*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)


colors = {
    "recon": "#2C7FB8",   
    "pred":  "#6A3D9A",   
    "total": "#D62728",  
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        is_train = "train" in col
        
        ax.plot(
            baseline_cera_noalign_history_df["epoch"],
            baseline_cera_noalign_history_df[col],
            linestyle="--" if is_train else "-", 
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title("CERA no-align training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")

ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)

ax.legend(title="Loss components", ncol=2, fontsize=9)

plt.show()